# ARC_ATLAS v4 Slice-Block Goal-0.8 Experiment

This notebook replaces the old axial 3-slice run with a higher-ceiling 2.5D setup aimed at the failure mode seen in the logs: small ATLAS lesions were under-detected or overcalled while large ARC lesions were already much better.

Design changes:

- 7-slice full-resolution blocks instead of 3-slice blocks.
- One whole brain per Keras step is preserved.
- Source + lesion-size balanced case sampling is enabled.
- Small-lesion brains get extra loss weight without making false positives free.
- Axial, sagittal, and coronal models are trained separately and then ensembled into one probability map.
- Whole-brain validation still writes per-epoch CSV/JSONL logs and NIfTI probability/segmentation outputs.


In [ ]:
from pathlib import Path
import sys
import time
import json

candidates = [
    Path.cwd(),
    Path.cwd() / "ARC_ATLAS_Combined" / "ARC_ATLAS_Train_v4",
    Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4"),
]
PROJECT_ROOT = next(
    (p for p in candidates if (p / "src" / "training_v2_slice_blocks.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate ARC_ATLAS_Train_v4/src/training_v2_slice_blocks.py")

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import training_v2_slice_blocks as seg

TRAIN_DIR = PROJECT_ROOT / "data" / "splits" / "90_10_random" / "train"
RUN_STAMP = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = PROJECT_ROOT / "runs" / f"{RUN_STAMP}_slice_blocks_triplanar_goal08"

AXIS_NAMES = {0: "sagittal", 1: "coronal", 2: "axial"}
AXES_TO_TRAIN = (2, 0, 1)  # axial first for direct comparison with the previous run, then sagittal/coronal for ensemble

COMMON_CFG = dict(
    DATA_DIR=TRAIN_DIR,
    IMAGES_DIR=TRAIN_DIR / "t1",
    MASKS_DIR=TRAIN_DIR / "masks",
    MANIFEST_PATH=TRAIN_DIR / "manifest.csv",
    TARGET_SHAPE=(192, 224, 192),
    RESAMPLE_TO_TARGET=False,
    BLOCK_DEPTH=7,
    SLICE_STRIDE=1,
    TOTAL_EPOCHS=90,
    INITIAL_LR=1.5e-4,
    MIN_LR=3e-6,
    WEIGHT_DECAY=1e-5,
    MAX_GRAD_NORM=1.0,
    MIXED_PRECISION=False,
    JIT_COMPILE=False,
    BASE_FILTERS=12,
    UNET_DEPTH=4,
    DROPOUT_RATE=0.08,
    L2_REG=1e-4,
    BALANCED_CASE_SAMPLING=True,
    SOURCE_BALANCED_SAMPLING=True,
    SIZE_AWARE_SAMPLING=True,
    SIZE_BUCKET_EDGES=(100, 1000, 10000),
    SIZE_BUCKET_PROBS=(0.32, 0.30, 0.23, 0.15),
    STEPS_PER_EPOCH=900,
    VALIDATION_STEPS=16,
    POSITIVE_WEIGHT=30.0,
    BCE_WEIGHT=0.36,
    DICE_WEIGHT=0.44,
    FOCAL_TVERSKY_WEIGHT=0.20,
    TVERSKY_ALPHA=0.55,
    TVERSKY_BETA=0.45,
    FOCAL_TVERSKY_GAMMA=1.33,
    LESION_SLICE_WEIGHT=1.35,
    EMPTY_SLICE_WEIGHT=0.90,
    DICE_ON_LESION_SLICES_ONLY=False,
    POSITIVE_TOPK_WEIGHT=0.045,
    POSITIVE_TOPK_FRACTION=0.35,
    SMALL_LESION_BOOST_REFERENCE=10000.0,
    SMALL_LESION_BOOST_MAX=2.5,
    AUGMENT=True,
    AUG_FLIP_PROB=0.5,
    AUG_INTENSITY_SCALE=0.08,
    AUG_INTENSITY_SHIFT=0.04,
    AUG_NOISE_STD=0.010,
    DECISION_THRESHOLD=0.25,
    VAL_THRESHOLD_SWEEP=(0.03, 0.05, 0.075, 0.10, 0.125, 0.15, 0.175, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.60, 0.70, 0.80),
    WHOLE_BRAIN_VAL_EVERY_N_EPOCHS=1,
    WHOLE_BRAIN_VAL_MAX_CASES=None,
    SAVE_VAL_PREDICTIONS=True,
    NUM_VAL_PREDICTIONS=4,
    EARLY_STOPPING_PATIENCE=18,
    EARLY_STOPPING_MIN_DELTA=0.001,
    RESTORE_BEST_WEIGHTS=True,
    TARGET_WHOLE_DICE=0.80,
    FIT_VERBOSE=2,
)

def make_axis_cfg(axis: int):
    name = AXIS_NAMES[axis]
    axis_dir = RUN_DIR / f"axis{axis}_{name}"
    return seg.SliceBlockTrainingConfig(
        **COMMON_CFG,
        SLICE_AXIS=axis,
        MODEL_DIR=axis_dir / "models",
        CALLBACKS_DIR=axis_dir / "callbacks",
    )

axis_cfgs = {axis: make_axis_cfg(axis) for axis in AXES_TO_TRAIN}

print(f"Project root: {PROJECT_ROOT}")
print(f"Training data: {TRAIN_DIR}")
print(f"Run dir: {RUN_DIR}")
for axis, cfg in axis_cfgs.items():
    print(f"axis {axis} ({AXIS_NAMES[axis]}): input per brain = (num_slices, {cfg.input_shape[0]}, {cfg.input_shape[1]}, {cfg.input_shape[2]}), checkpoint = {cfg.checkpoint_path}")
print(f"Target whole-brain Dice stop: {COMMON_CFG['TARGET_WHOLE_DICE']}")


In [ ]:
# Check split composition and the exact source/lesion-size problem this run is targeting.
import pandas as pd

base_cfg = axis_cfgs[2 if 2 in axis_cfgs else AXES_TO_TRAIN[0]]
cases = seg.load_cases(base_cfg)
train_cases, val_cases, lesion_sizes = seg.split_cases(cases, base_cfg)
size_by_id = {case.case_id: size for case, size in zip(cases, lesion_sizes)}

def split_frame(split_name, split_cases):
    return pd.DataFrame([
        {
            "split": split_name,
            "source": case.source,
            "case_id": case.case_id,
            "lesion_voxels": int(size_by_id[case.case_id]),
            "lesion_group": seg.lesion_size_group(int(size_by_id[case.case_id])),
        }
        for case in split_cases
    ])

split_df = pd.concat([split_frame("train", train_cases), split_frame("val", val_cases)], ignore_index=True)
display(split_df.groupby(["split", "source"]).size().unstack(fill_value=0))
display(split_df.groupby(["split", "lesion_group"]).size().unstack(fill_value=0))
display(split_df.groupby(["source", "lesion_group"]).size().unstack(fill_value=0))


In [ ]:
# Sanity check one whole-brain 7-slice batch and model size before launching training.
sanity_cfg = base_cfg
image, mask, _ = seg.load_case_arrays(cases[0], sanity_cfg)
x, y = seg.make_slice_blocks(image, mask, sanity_cfg)
model = seg.build_slice_block_model(sanity_cfg)

print(f"Cases: {len(cases)}")
print(f"Prepared brain: image={image.shape}, mask={mask.shape}")
print(f"One-brain batch: x={x.shape}, y={y.shape}")
print(f"Lesion voxels in sanity case: {int(y.sum())}")
print(f"Model params: {model.count_params():,}")

# Quick loss/shape smoke test without consuming a real epoch.
model.compile(optimizer="adam", loss=seg.WeightedBceDiceTversky(
    positive_weight=sanity_cfg.POSITIVE_WEIGHT,
    bce_weight=sanity_cfg.BCE_WEIGHT,
    dice_weight=sanity_cfg.DICE_WEIGHT,
    focal_tversky_weight=sanity_cfg.FOCAL_TVERSKY_WEIGHT,
    tversky_alpha=sanity_cfg.TVERSKY_ALPHA,
    tversky_beta=sanity_cfg.TVERSKY_BETA,
    focal_tversky_gamma=sanity_cfg.FOCAL_TVERSKY_GAMMA,
    lesion_slice_weight=sanity_cfg.LESION_SLICE_WEIGHT,
    empty_slice_weight=sanity_cfg.EMPTY_SLICE_WEIGHT,
    positive_topk_weight=sanity_cfg.POSITIVE_TOPK_WEIGHT,
    positive_topk_fraction=sanity_cfg.POSITIVE_TOPK_FRACTION,
    small_lesion_boost_reference=sanity_cfg.SMALL_LESION_BOOST_REFERENCE,
    small_lesion_boost_max=sanity_cfg.SMALL_LESION_BOOST_MAX,
))
loss_value = model.train_on_batch(x[:2], y[:2])
print(f"Smoke-test loss on 2 slices: {float(loss_value):.6f}")

del model
seg.tf.keras.backend.clear_session()


In [ ]:
# Visual sanity check for one 7-slice input block and center-slice target.
import matplotlib.pyplot as plt
import numpy as np

center_idx = int(np.argmax(y.reshape(y.shape[0], -1).sum(axis=1)))
block = x[center_idx]
target = y[center_idx, ..., 0]
center_channel = sanity_cfg.BLOCK_DEPTH // 2

fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for channel, ax in enumerate(axes.flat[:sanity_cfg.BLOCK_DEPTH]):
    ax.imshow(block[..., channel].T, cmap="gray", origin="lower")
    ax.set_title(f"Input channel {channel + 1}")
    ax.axis("off")
axes.flat[-1].imshow(block[..., center_channel].T, cmap="gray", origin="lower")
if target.any():
    axes.flat[-1].contour(target.T, levels=[0.5], colors=["red"], linewidths=1.0)
axes.flat[-1].set_title("Center target")
axes.flat[-1].axis("off")
plt.tight_layout()


In [ ]:
# Launch staged training. Each axis writes its own epoch logs, checkpoints, and whole-brain predictions.
# This can run for a long time. If interrupted after one axis, rerun from this cell after editing AXES_TO_TRAIN above to the remaining axes.
histories = {}
for axis, cfg in axis_cfgs.items():
    print("=" * 100)
    print(f"Training axis {axis} ({AXIS_NAMES[axis]})")
    print(f"Run callbacks: {cfg.CALLBACKS_DIR}")
    histories[axis] = seg.train_slice_block_model(cfg)
    seg.tf.keras.backend.clear_session()
    print(f"Finished axis {axis}. Best checkpoint: {cfg.checkpoint_path}")


In [ ]:
# Review per-axis epoch logs and source/lesion-group breakdowns.
import json
import pandas as pd

for axis, cfg in axis_cfgs.items():
    print("=" * 100)
    print(f"axis {axis} ({AXIS_NAMES[axis]})")
    summary_path = cfg.CALLBACKS_DIR / "whole_val_summary.jsonl"
    train_log = cfg.CALLBACKS_DIR / "training_log.csv"
    if train_log.exists():
        display(pd.read_csv(train_log).tail(8))
    else:
        print(f"No training log yet: {train_log}")
    if not summary_path.exists():
        print(f"No whole-brain summary yet: {summary_path}")
        continue
    rows = [json.loads(line) for line in summary_path.read_text().splitlines() if line.strip()]
    summary_df = pd.DataFrame(rows)
    display(summary_df.tail(8))
    best_epoch = int(summary_df.loc[summary_df["val_whole_dice_hard_best_thr_score"].idxmax(), "epoch"])
    best_csv = cfg.CALLBACKS_DIR / f"whole_val_epoch_{best_epoch:04d}.csv"
    print(f"best epoch: {best_epoch}, csv: {best_csv}")
    if best_csv.exists():
        case_df = pd.read_csv(best_csv)
        display(case_df.groupby("source")["best_threshold_dice"].agg(["count", "mean", "median"]))
        display(case_df.groupby("lesion_group")["best_threshold_dice"].agg(["count", "mean", "median"]))


In [ ]:
# Evaluate the tri-planar probability-map ensemble after at least one axis has produced best weights.
trained_axes = [axis for axis, cfg in axis_cfgs.items() if cfg.checkpoint_path.exists()]
if not trained_axes:
    raise FileNotFoundError("No axis checkpoint exists yet. Run the training cell first.")

trained_cfgs = [axis_cfgs[axis] for axis in trained_axes]
weights_paths = [axis_cfgs[axis].checkpoint_path for axis in trained_axes]
ensemble_dir = RUN_DIR / "ensemble"

summary = seg.evaluate_slice_block_ensemble(
    configs=trained_cfgs,
    weights_paths=weights_paths,
    out_dir=ensemble_dir,
    thresholds=COMMON_CFG["VAL_THRESHOLD_SWEEP"],
    decision_threshold=COMMON_CFG["DECISION_THRESHOLD"],
    save_predictions=6,
)

print(json.dumps(summary, indent=2))
print(f"Ensemble CSV: {ensemble_dir / 'ensemble_val.csv'}")
print(f"Ensemble predictions: {ensemble_dir / 'predictions'}")


In [ ]:
# Export one manual ensemble probability map and threshold segmentation for visual inspection.
trained_axes = [axis for axis, cfg in axis_cfgs.items() if cfg.checkpoint_path.exists()]
if not trained_axes:
    raise FileNotFoundError("No axis checkpoint exists yet. Run the training cell first.")

models = [seg.load_slice_block_model(axis_cfgs[axis], axis_cfgs[axis].checkpoint_path) for axis in trained_axes]
trained_cfgs = [axis_cfgs[axis] for axis in trained_axes]
case = val_cases[0]
prob, mask, ref_img = seg.predict_ensemble_probability_map(models, trained_cfgs, case)
out_dir = RUN_DIR / "manual_ensemble_prediction"
seg.save_case_outputs(out_dir, case, prob, COMMON_CFG["DECISION_THRESHOLD"], ref_img)

print(f"Saved probability map and threshold segmentation to: {out_dir}")
print(f"Case: {case.case_id}")
print(f"Probability range: min={float(prob.min()):.6f}, max={float(prob.max()):.6f}")
print(f"Threshold voxels @ {COMMON_CFG['DECISION_THRESHOLD']:.2f}: {int((prob >= COMMON_CFG['DECISION_THRESHOLD']).sum())}")
seg.tf.keras.backend.clear_session()
